# Amazon section: educational churn proxy workflow

This workflow uses generic customer-churn proxy data, not Amazon Prime Video customer records. The label-generation rule, source provenance, and feature timing are unverified. Reported metrics are educational proxy-dataset results only.

The supplied test CSV is audited for ID and label conflicts but is never combined with training data and never used for model evaluation. The single all-null training row is documented and excluded in memory only. CustomerID is excluded from model inputs.


In [1]:
from pathlib import Path
import io
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

ROOT = Path(r"C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard")
TRAIN_PATH = ROOT / "Datasets/Amazon/customer_churn_dataset-training-master.csv"
TEST_PATH = ROOT / "Datasets/Amazon/customer_churn_dataset-testing-master.csv"
OUTPUTS = {
    "model": ROOT / "models/amazon_proxy_churn_model.pkl",
    "quality": ROOT / "reports/amazon_proxy_data_quality_summary.csv",
    "overlap": ROOT / "reports/amazon_proxy_train_test_overlap_audit.csv",
    "comparison": ROOT / "reports/amazon_proxy_model_comparison.csv",
    "provenance": ROOT / "reports/amazon_proxy_provenance_and_scope.md",
}
EXPECTED_COLUMNS = ["CustomerID", "Age", "Gender", "Tenure", "Usage Frequency",
    "Support Calls", "Payment Delay", "Subscription Type", "Contract Length",
    "Total Spend", "Last Interaction", "Churn"]


In [2]:
# Load both source files for inspection. The testing file is not a model evaluation set.
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
assert list(train_raw.columns) == EXPECTED_COLUMNS
assert list(test_raw.columns) == EXPECTED_COLUMNS
quality_rows = []
for name, df in [("training", train_raw), ("testing_not_used_for_evaluation", test_raw)]:
    quality_rows.extend([
        {"dataset": name, "check": "rows", "field": "", "value": len(df), "note": ""},
        {"dataset": name, "check": "columns", "field": "", "value": len(df.columns), "note": ""},
        {"dataset": name, "check": "missing_cells", "field": "", "value": int(df.isna().sum().sum()), "note": ""},
        {"dataset": name, "check": "all_null_rows", "field": "", "value": int(df.isna().all(axis=1).sum()), "note": "Training all-null row is excluded in memory only." if name == "training" else ""},
        {"dataset": name, "check": "duplicate_full_rows", "field": "", "value": int(df.duplicated().sum()), "note": ""},
        {"dataset": name, "check": "duplicate_customer_ids", "field": "CustomerID", "value": int(df.CustomerID.duplicated().sum()), "note": ""},
    ])
    for col, dtype in df.dtypes.astype(str).items():
        quality_rows.append({"dataset": name, "check": "dtype", "field": col, "value": dtype, "note": ""})
    for label, count in df.Churn.value_counts(dropna=False).items():
        label_text = "missing" if pd.isna(label) else str(int(label))
        quality_rows.append({"dataset": name, "check": "target_distribution", "field": "Churn=" + label_text, "value": int(count), "note": "Operational target definition is unverified."})
quality_df = pd.DataFrame(quality_rows)


In [3]:
# Aggregate overlap audit only; individual customer IDs are never written to a report.
train_ids = train_raw.dropna(subset=["CustomerID"]).copy()
test_ids = test_raw.dropna(subset=["CustomerID"]).copy()
train_ids["CustomerID"] = train_ids.CustomerID.astype("int64")
test_ids["CustomerID"] = test_ids.CustomerID.astype("int64")
shared = train_ids.merge(test_ids, on="CustomerID", how="inner",
                         suffixes=("_train", "_test"), validate="one_to_one")
audit_rows = []
def audit(check, value, note=""):
    audit_rows.append({"check": check, "value": value, "note": note})
audit("training_unique_ids", int(train_ids.CustomerID.nunique()))
audit("testing_unique_ids", int(test_ids.CustomerID.nunique()))
audit("shared_unique_ids", len(shared), "Individual IDs omitted.")
audit("shared_ids_percent_of_testing_ids", round(100 * len(shared) / len(test_ids), 4))
labels_match = shared.Churn_train.eq(shared.Churn_test)
audit("shared_ids_matching_labels", int(labels_match.sum()))
audit("shared_ids_conflicting_labels", int((~labels_match).sum()), "Conflicts are reported, not discarded.")
features = [c for c in EXPECTED_COLUMNS if c not in ("CustomerID", "Churn")]
same_features = np.ones(len(shared), dtype=bool)
for col in features:
    eq = shared[col + "_train"].eq(shared[col + "_test"]) | (shared[col + "_train"].isna() & shared[col + "_test"].isna())
    audit("shared_ids_feature_differences:" + col, int((~eq).sum()))
    same_features &= eq.to_numpy()
audit("shared_ids_with_any_feature_difference", int((~same_features).sum()))
audit("identical_full_records_across_files", int((same_features & labels_match.to_numpy()).sum()))
audit("testing_ids_not_in_training", len(test_ids) - len(shared))
audit("training_ids_not_in_testing", len(train_ids) - len(shared))
overlap_df = pd.DataFrame(audit_rows)
print("Training shape:", train_raw.shape, "| testing shape:", test_raw.shape)
print("Training missing cells:", int(train_raw.isna().sum().sum()), "| all-null rows:", int(train_raw.isna().all(axis=1).sum()))
print("Within-file duplicate IDs:", int(train_raw.CustomerID.duplicated().sum()), "train /", int(test_raw.CustomerID.duplicated().sum()), "test")
print("Shared IDs:", len(shared), "| conflicting labels:", int((~labels_match).sum()), "| all shared IDs have feature changes:", bool((~same_features).all()))
print("Testing CSV is audit-only; it is not combined or used for evaluation.")


Training shape: (440833, 12) | testing shape: (64374, 12)
Training missing cells: 12 | all-null rows: 1
Within-file duplicate IDs: 0 train / 0 test
Shared IDs: 62995 | conflicting labels: 32778 | all shared IDs have feature changes: True
Testing CSV is audit-only; it is not combined or used for evaluation.


In [4]:
# Handle exactly the fully empty training row in memory, validate the target, and prepare features.
all_null_rows = int(train_raw.isna().all(axis=1).sum())
if all_null_rows != 1:
    raise ValueError(f"Expected exactly one all-null training row; found {all_null_rows}.")
model_data = train_raw.loc[~train_raw.isna().all(axis=1)].copy()
if model_data.Churn.isna().any() or set(model_data.Churn.unique()) != {0, 1}:
    raise ValueError("Churn must be complete and binary 0/1 after all-null row handling.")
if model_data.CustomerID.isna().any() or model_data.CustomerID.duplicated().any():
    raise ValueError("Training CustomerID validation failed.")
X = model_data.drop(columns=["CustomerID", "Churn"])
y = model_data.Churn.astype("int64")
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
print("Explicitly excluded all-null rows:", all_null_rows)
print("CustomerID excluded from features:", "CustomerID" not in X.columns)
print("Feature columns:", list(X.columns))
print("Stratified split: train", len(X_train), "| holdout", len(X_holdout), "| random_state=42")
print("Training target counts:", y_train.value_counts().sort_index().to_dict())
print("Holdout target counts:", y_holdout.value_counts().sort_index().to_dict())


Explicitly excluded all-null rows: 1
CustomerID excluded from features: True
Feature columns: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']
Stratified split: train 352665 | holdout 88167 | random_state=42
Training target counts: {0: 152666, 1: 199999}
Holdout target counts: {0: 38167, 1: 50000}


In [5]:
# Each complete pipeline fits imputation, scaling, and encoding on training rows only.
def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_cols),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ])

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", make_preprocessor(True)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", make_preprocessor(False)),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
}
metric_rows = []
predictions = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_holdout)
    predictions[name] = pred
    tn, fp, fn, tp = confusion_matrix(y_holdout, pred, labels=[0, 1]).ravel()
    metric_rows.append({
        "dataset": "Generic customer churn educational proxy",
        "model": name,
        "split": "Stratified 80/20 from nonempty training CSV; random_state=42",
        "positive_class": "Churn=1",
        "train_rows": len(X_train), "holdout_rows": len(X_holdout),
        "accuracy": accuracy_score(y_holdout, pred),
        "precision_churn_1": precision_score(y_holdout, pred, pos_label=1, zero_division=0),
        "recall_churn_1": recall_score(y_holdout, pred, pos_label=1, zero_division=0),
        "f1_churn_1": f1_score(y_holdout, pred, pos_label=1, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "scope_note": "Educational proxy-dataset result only; not Amazon Prime Video performance. Provenance, target definition, and feature timing are unverified."
    })
    print("\n", name)
    print("Accuracy:", accuracy_score(y_holdout, pred))
    print("Churn precision:", precision_score(y_holdout, pred, pos_label=1, zero_division=0))
    print("Churn recall:", recall_score(y_holdout, pred, pos_label=1, zero_division=0))
    print("Churn F1:", f1_score(y_holdout, pred, pos_label=1, zero_division=0))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_holdout, pred, labels=[0, 1]))
comparison_df = pd.DataFrame(metric_rows)
best = max(metric_rows, key=lambda row: (row["f1_churn_1"], row["recall_churn_1"], row["accuracy"]))
selected_name = best["model"]
selected_pipeline = models[selected_name]
comparison_df["selected_model"] = comparison_df.model.eq(selected_name)
comparison_df["selection_rule"] = "Highest churn-class F1; tie-break by churn recall, then accuracy"
comparison_df["saved_model"] = comparison_df.model.eq(selected_name).map({True: OUTPUTS["model"].name, False: ""})
print("\nSelected:", selected_name)



 Logistic Regression
Accuracy: 0.8933728038835392
Churn precision: 0.9233560658199337
Churn recall: 0.88548
Churn F1: 0.9040214805664172
Confusion matrix [[TN, FP], [FN, TP]]:
[[34492  3675]
 [ 5726 44274]]

 Random Forest
Accuracy: 0.9992627627116721
Churn precision: 0.9999399291163573
Churn recall: 0.99876
Churn F1: 0.9993496162736014
Confusion matrix [[TN, FP], [FN, TP]]:
[[38164     3]
 [   62 49938]]

Selected: Random Forest


In [6]:
# Read-only reproducibility check against the already-saved model and comparison report.
saved_model = joblib.load(OUTPUTS["model"])
saved_pred = saved_model.predict(X_holdout)
fresh_rf_pred = predictions["Random Forest"]
assert np.array_equal(saved_pred, fresh_rf_pred), "Fresh RF predictions differ from the saved model."
saved_report = pd.read_csv(OUTPUTS["comparison"])
for _, row in comparison_df.iterrows():
    old = saved_report.loc[saved_report.model == row.model].iloc[0]
    for col in ["accuracy", "precision_churn_1", "recall_churn_1", "f1_churn_1"]:
        assert np.isclose(float(row[col]), float(old[col]), rtol=0, atol=1e-12), f"Metric mismatch: {row.model} {col}"
    for col in ["tn", "fp", "fn", "tp"]:
        assert int(row[col]) == int(old[col]), f"Confusion count mismatch: {row.model} {col}"
print("PASS: freshly trained Random Forest predictions match the saved model on the same holdout.")
print("PASS: both models' metrics and confusion counts match the saved comparison report.")


PASS: freshly trained Random Forest predictions match the saved model on the same holdout.
PASS: both models' metrics and confusion counts match the saved comparison report.


## Output safety

The final cell checks all five approved model/report paths again. If any exists, it reports the conflict and writes nothing. It never overwrites a model or report. The notebook itself is the only file updated when this notebook is authored.


In [7]:
# Save only if all five output paths are absent. Existing outputs cause a safe stop.
scope_text = '''# Amazon section: educational proxy model

This workflow uses generic customer-churn proxy data, not Amazon Prime Video customer records. The data origin, operational Churn definition, outcome window, and feature timing are unverified. All metrics are educational proxy-dataset results only.

The supplied testing CSV is not combined with training data and is not used for evaluation because 62,995 IDs overlap and 32,778 shared IDs have conflicting labels. The one all-null training row is documented and excluded in memory only. CustomerID is excluded from model inputs.

Evaluation uses a reproducible stratified 80/20 holdout from nonempty training rows (random_state=42). Preprocessing is fitted within each estimator pipeline on training rows only. Near-perfect Random Forest performance is not verified Amazon Prime Video performance and may reflect unverified label or feature construction.
'''
buffer = io.BytesIO()
joblib.dump(selected_pipeline, buffer, compress=3)
payloads = {
    OUTPUTS["model"]: buffer.getvalue(),
    OUTPUTS["quality"]: quality_df.to_csv(index=False).encode("utf-8"),
    OUTPUTS["overlap"]: overlap_df.to_csv(index=False).encode("utf-8"),
    OUTPUTS["comparison"]: comparison_df.to_csv(index=False).encode("utf-8"),
    OUTPUTS["provenance"]: scope_text.encode("utf-8"),
}
conflicts = [str(path) for path in payloads if path.exists()]
if conflicts:
    print("SAVE STOPPED. Existing outputs would be replaced; no files were written:")
    print("\n".join(conflicts))
else:
    for path, content in payloads.items():
        with path.open("xb") as handle:
            handle.write(content)
    print("Created the five approved model/report files with exclusive-create mode.")


SAVE STOPPED. Existing outputs would be replaced; no files were written:
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\models\amazon_proxy_churn_model.pkl
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\reports\amazon_proxy_data_quality_summary.csv
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\reports\amazon_proxy_train_test_overlap_audit.csv
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\reports\amazon_proxy_model_comparison.csv
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\reports\amazon_proxy_provenance_and_scope.md
